In [1]:
#1
import base64
import io
import os
import textwrap
import threading
import uuid
import webbrowser
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from flask import Flask, request, render_template_string
from werkzeug.utils import secure_filename


app = Flask(__name__)
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()
UPLOAD_CACHE_DIR = os.path.join(BASE_DIR, "uploaded_datasets")


@dataclass
class RuleNode:
    node_id: int
    depth: int
    row_indices: List[int]
    conditions: List[str] = field(default_factory=list)
    split_feature: Optional[str] = None
    split_operator: Optional[str] = None
    split_value: Optional[object] = None
    missing_goes_left: bool = False
    left: Optional["RuleNode"] = None
    right: Optional["RuleNode"] = None

    @property
    def is_leaf(self) -> bool:
        return self.left is None and self.right is None


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = (
        cleaned.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    return cleaned


def save_uploaded_dataset(uploaded_file) -> str:
    os.makedirs(UPLOAD_CACHE_DIR, exist_ok=True)
    original_name = secure_filename(uploaded_file.filename)
    _, extension = os.path.splitext(original_name)
    extension = extension.lower()

    if extension not in {".csv", ".xlsx", ".xls"}:
        raise ValueError("Only CSV, XLSX, and XLS files are supported.")

    dataset_id = f"{uuid.uuid4().hex}{extension}"
    path = os.path.join(UPLOAD_CACHE_DIR, dataset_id)
    uploaded_file.save(path)
    return dataset_id


def get_cached_dataset_path(dataset_id: str) -> str:
    safe_dataset_id = secure_filename(dataset_id)
    path = os.path.join(UPLOAD_CACHE_DIR, safe_dataset_id)
    if not os.path.exists(path):
        raise ValueError("Uploaded dataset was not found. Please upload the file again.")
    return path


def read_dataset(source) -> pd.DataFrame:
    filename = source if isinstance(source, str) else source.filename
    filename = filename.lower()

    if filename.endswith(".csv"):
        return pd.read_csv(source)
    if filename.endswith((".xlsx", ".xls")):
        return pd.read_excel(source)

    raise ValueError("Only CSV, XLSX, and XLS files are supported.")


def load_clean_dataset(dataset_id: str) -> pd.DataFrame:
    return clean_column_names(read_dataset(get_cached_dataset_path(dataset_id)))


def get_dataset_columns(dataset_id: str) -> List[str]:
    return load_clean_dataset(dataset_id).columns.tolist()


def get_dataset_preview(dataset_id: str) -> str:
    return load_clean_dataset(dataset_id).head(8).to_html(index=False, classes="table")


def identify_variable_types(
    df: pd.DataFrame,
    target_col: str,
    categorical_unique_threshold: int,
) -> Tuple[List[str], List[str], List[str]]:
    features = df.drop(columns=[target_col])
    continuous_cols = []
    numeric_cols = []
    categorical_cols = []

    for col in features.columns:
        series = features[col]
        is_numeric = pd.api.types.is_numeric_dtype(series)
        is_bool = pd.api.types.is_bool_dtype(series)

        if is_numeric or is_bool:
            unique_count = series.nunique(dropna=True)
            if is_bool or unique_count <= categorical_unique_threshold:
                numeric_cols.append(col)
            else:
                continuous_cols.append(col)
        else:
            categorical_cols.append(col)

    return continuous_cols, categorical_cols, numeric_cols


def cap_outliers(
    df: pd.DataFrame,
    continuous_cols: List[str],
    lower_cap_value: float,
    upper_cap_value: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    capped = df.copy()
    report_rows = []

    lower_q = max(0.0, min(1.0, lower_cap_value / 100.0))
    upper_q = max(0.0, min(1.0, upper_cap_value / 100.0))

    for col in continuous_cols:
        numeric_series = pd.to_numeric(capped[col], errors="coerce")
        lower_cap = numeric_series.quantile(lower_q)
        upper_cap = numeric_series.quantile(upper_q)

        if pd.notna(lower_cap) and pd.notna(upper_cap) and lower_cap > upper_cap:
            lower_cap, upper_cap = upper_cap, lower_cap

        low_count = int((numeric_series < lower_cap).sum()) if pd.notna(lower_cap) else 0
        high_count = int((numeric_series > upper_cap).sum()) if pd.notna(upper_cap) else 0
        capped[col] = numeric_series.clip(lower=lower_cap, upper=upper_cap)

        report_rows.append(
            {
                "variable": col,
                "lower_percentile": lower_cap_value,
                "upper_percentile": upper_cap_value,
                "applied_lower_cap": round(float(lower_cap), 6) if pd.notna(lower_cap) else np.nan,
                "applied_upper_cap": round(float(upper_cap), 6) if pd.notna(upper_cap) else np.nan,
                "rows_capped_low": low_count,
                "rows_capped_high": high_count,
            }
        )

    if not report_rows:
        report_rows.append(
            {
                "variable": "No continuous variables",
                "lower_percentile": "",
                "upper_percentile": "",
                "applied_lower_cap": "",
                "applied_upper_cap": "",
                "rows_capped_low": "",
                "rows_capped_high": "",
            }
        )

    return capped, pd.DataFrame(report_rows)


def make_table(values: List[str], column_name: str) -> str:
    if not values:
        values = ["None"]
    return pd.DataFrame({column_name: values}).to_html(index=False, classes="table")


def text_to_base64(value: str) -> str:
    return base64.b64encode(value.encode("utf-8")).decode("utf-8")


def infer_good_bad_class_indices(
    class_names: List[str],
    target_col: Optional[str] = None,
    target_meaning: str = "auto",
) -> Tuple[Optional[int], Optional[int]]:
    normalized = [str(name).strip().lower().replace("_", " ") for name in class_names]

    high_values = {"1", "yes", "y", "true", "survived", "alive", "approved", "success", "good"}
    low_values = {"0", "no", "n", "false", "died", "dead", "rejected", "fail", "failed", "bad"}
    high_index = next((idx for idx, name in enumerate(normalized) if name in high_values), None)
    low_index = next((idx for idx, name in enumerate(normalized) if name in low_values), None)

    if len(class_names) == 2 and high_index is not None and low_index is not None:
        if target_meaning == "high_good":
            return high_index, low_index
        if target_meaning == "high_bad":
            return low_index, high_index

    bad_terms = {"bad", "default", "defaulter", "fail", "failed", "npa", "yes", "y", "1", "true"}
    good_terms = {"good", "non default", "no", "n", "0", "false", "performing"}
    bad_index = next((idx for idx, name in enumerate(normalized) if name in bad_terms), None)
    good_index = next((idx for idx, name in enumerate(normalized) if name in good_terms), None)

    if len(class_names) == 2:
        if bad_index is not None and good_index is None:
            good_index = 1 - bad_index
        elif good_index is not None and bad_index is None:
            bad_index = 1 - good_index
        elif good_index is None and bad_index is None:
            good_index = 0
            bad_index = 1

    return good_index, bad_index


def risk_band_from_bad_rate(bad_rate: Optional[float]) -> str:
    if bad_rate is None:
        return "N/A"
    if bad_rate < 0.33:
        return "Low Risk"
    if bad_rate < 0.66:
        return "Medium Risk"
    return "High Risk"


def gini_impurity(values: pd.Series) -> float:
    if values.empty:
        return 0.0
    proportions = values.astype(str).value_counts(normalize=True)
    return 1.0 - float((proportions * proportions).sum())


def split_score(left_targets: pd.Series, right_targets: pd.Series) -> float:
    total = len(left_targets) + len(right_targets)
    if total == 0 or len(left_targets) == 0 or len(right_targets) == 0:
        return -1.0

    before = gini_impurity(pd.concat([left_targets, right_targets]))
    after = (len(left_targets) / total) * gini_impurity(left_targets) + (
        len(right_targets) / total
    ) * gini_impurity(right_targets)
    return before - after


def split_candidate_thresholds(values: pd.Series, max_candidates: int = 80) -> List[float]:
    unique_values = np.sort(pd.to_numeric(values, errors="coerce").dropna().unique())
    if len(unique_values) < 2:
        return []

    midpoints = (unique_values[:-1] + unique_values[1:]) / 2.0
    if len(midpoints) <= max_candidates:
        return [float(value) for value in midpoints]

    positions = np.linspace(0, len(midpoints) - 1, max_candidates).round().astype(int)
    return [float(midpoints[pos]) for pos in np.unique(positions)]


def best_split_for_column(df: pd.DataFrame, target_col: str, col: str, row_indices: List[int]):
    subset = df.loc[row_indices, [col, target_col]]
    if subset.empty or subset[col].nunique(dropna=True) < 2:
        return None

    series = subset[col]
    if pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
        best = None
        missing_idx = subset.index[numeric.isna()].tolist()
        non_missing_idx = subset.index[numeric.notna()].tolist()

        for threshold in split_candidate_thresholds(numeric):
            left_base = subset.index[numeric <= threshold].tolist()
            right_base = subset.index[numeric > threshold].tolist()
            for missing_goes_left in [False, True]:
                left_idx = left_base + (missing_idx if missing_goes_left else [])
                right_idx = right_base + ([] if missing_goes_left else missing_idx)
                if not left_idx or not right_idx or len(left_idx) + len(right_idx) != len(row_indices):
                    continue
                score = split_score(df.loc[left_idx, target_col], df.loc[right_idx, target_col])
                if best is None or score > best["score"]:
                    best = {
                        "score": score,
                        "feature": col,
                        "operator": "<=",
                        "value": float(threshold),
                        "left_idx": left_idx,
                        "right_idx": right_idx,
                        "missing_goes_left": missing_goes_left,
                    }

        if best is None and missing_idx and non_missing_idx:
            left_idx = missing_idx
            right_idx = non_missing_idx
            score = split_score(df.loc[left_idx, target_col], df.loc[right_idx, target_col])
            best = {
                "score": score,
                "feature": col,
                "operator": "is missing",
                "value": "",
                "left_idx": left_idx,
                "right_idx": right_idx,
                "missing_goes_left": True,
            }
        return best

    categorical = series.astype("object").where(series.notna(), "__MISSING__").astype(str)
    counts = categorical.value_counts()
    candidates = counts.index.tolist()
    best = None
    for value in candidates:
        left_idx = subset.index[categorical == value].tolist()
        right_idx = subset.index[categorical != value].tolist()
        if not left_idx or not right_idx or len(left_idx) + len(right_idx) != len(row_indices):
            continue
        score = split_score(df.loc[left_idx, target_col], df.loc[right_idx, target_col])
        if best is None or score > best["score"]:
            best = {
                "score": score,
                "feature": col,
                "operator": "==",
                "value": value,
                "left_idx": left_idx,
                "right_idx": right_idx,
                "missing_goes_left": value == "__MISSING__",
            }
    return best


def format_value(value) -> str:
    if isinstance(value, float):
        if abs(value) >= 100:
            return f"{value:.0f}"
        if abs(value) >= 10:
            return f"{value:.2f}".rstrip("0").rstrip(".")
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)


def build_rule_tree(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    max_depth: int,
    max_leaf_nodes: int,
) -> RuleNode:
    node_counter = {"value": 0}
    leaf_counter = {"value": 1}

    def next_node_id() -> int:
        node_counter["value"] += 1
        return node_counter["value"]

    def recurse(row_indices: List[int], depth: int, conditions: List[str]) -> RuleNode:
        node = RuleNode(next_node_id(), depth, row_indices, conditions)
        if depth >= max_depth or leaf_counter["value"] >= max_leaf_nodes or len(row_indices) < 4:
            return node

        best = None
        for col in feature_cols:
            candidate = best_split_for_column(df, target_col, col, row_indices)
            if candidate and (best is None or candidate["score"] > best["score"]):
                best = candidate

        if not best or best["score"] <= 0 or not best["left_idx"] or not best["right_idx"]:
            return node

        leaf_counter["value"] += 1
        node.split_feature = best["feature"]
        node.split_operator = best["operator"]
        node.split_value = best["value"]
        node.missing_goes_left = best.get("missing_goes_left", False)

        if best["operator"] == "is missing":
            left_condition = f"{best['feature']} is missing"
            right_condition = f"{best['feature']} is not missing"
        elif best["operator"] == "<=":
            left_condition = f"{best['feature']} <= {format_value(best['value'])}"
            right_condition = f"{best['feature']} > {format_value(best['value'])}"
            if best.get("missing_goes_left", False):
                left_condition = f"{left_condition} OR {best['feature']} is missing"
            else:
                right_condition = f"{right_condition} OR {best['feature']} is missing"
        elif best["operator"] == "==" and best["value"] == "__MISSING__":
            left_condition = f"{best['feature']} is missing"
            right_condition = f"{best['feature']} is not missing"
        else:
            right_condition = f"{best['feature']} != {format_value(best['value'])}"
            left_condition = f"{best['feature']} {best['operator']} {format_value(best['value'])}"

        node.left = recurse(best["left_idx"], depth + 1, conditions + [left_condition])
        node.right = recurse(best["right_idx"], depth + 1, conditions + [right_condition])
        return node

    return recurse(df.index.tolist(), 0, [])


def iter_leaves(root: RuleNode) -> List[RuleNode]:
    leaves = []

    def walk(node: RuleNode) -> None:
        if node.is_leaf:
            leaves.append(node)
            return
        walk(node.left)
        walk(node.right)

    walk(root)
    return leaves


def node_target_stats(
    df: pd.DataFrame,
    node: RuleNode,
    target_col: str,
    class_names: List[str],
    good_index: Optional[int],
    bad_index: Optional[int],
) -> Dict[str, object]:
    target = df.loc[node.row_indices, target_col].astype(str)
    counts = target.value_counts()
    total = int(len(target))
    predicted = str(counts.index[0]) if total else "N/A"
    good_rate = None
    bad_rate = None

    if good_index is not None and good_index < len(class_names) and total:
        good_rate = float((target == class_names[good_index]).sum() / total)
    if bad_index is not None and bad_index < len(class_names) and total:
        bad_rate = float((target == class_names[bad_index]).sum() / total)

    return {
        "rows": total,
        "predicted_class": predicted,
        "good_rate": good_rate,
        "bad_rate": bad_rate,
        "risk_band": risk_band_from_bad_rate(bad_rate),
        "distribution": {name: int(counts.get(name, 0)) for name in class_names},
    }


def make_leaf_summary_table(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    class_names = sorted(df[target_col].astype(str).unique().tolist())
    good_index, bad_index = infer_good_bad_class_indices(class_names, target_col, target_meaning)
    total_rows = len(df)
    rows = []

    for leaf_number, leaf in enumerate(iter_leaves(root), start=1):
        stats = node_target_stats(df, leaf, target_col, class_names, good_index, bad_index)
        rows.append(
            {
                "leaf_node": leaf_number,
                "risk_band": stats["risk_band"],
                "predicted_class": stats["predicted_class"],
                "rows": stats["rows"],
                "row_share": f"{stats['rows'] / total_rows:.1%}" if total_rows else "0.0%",
                "good_rate": "N/A" if stats["good_rate"] is None else f"{stats['good_rate']:.1%}",
                "bad_rate": "N/A" if stats["bad_rate"] is None else f"{stats['bad_rate']:.1%}",
            }
        )

    return pd.DataFrame(rows).to_html(index=False, classes="table")


def collect_leaf_paths(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> List[dict]:
    class_names = sorted(df[target_col].astype(str).unique().tolist())
    good_index, bad_index = infer_good_bad_class_indices(class_names, target_col, target_meaning)
    total_rows = len(df)
    rows = []

    for leaf_number, leaf in enumerate(iter_leaves(root), start=1):
        stats = node_target_stats(df, leaf, target_col, class_names, good_index, bad_index)
        rows.append(
            {
                "leaf_node": leaf_number,
                "if_conditions": " AND ".join(leaf.conditions) if leaf.conditions else "All rows",
                "final_decision": stats["predicted_class"],
                "risk_band": stats["risk_band"],
                "rows": stats["rows"],
                "row_share": f"{stats['rows'] / total_rows:.1%}" if total_rows else "0.0%",
                "good_rate": "N/A" if stats["good_rate"] is None else f"{stats['good_rate']:.1%}",
                "bad_rate": "N/A" if stats["bad_rate"] is None else f"{stats['bad_rate']:.1%}",
            }
        )
    return rows


def export_rules_text(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    rows = collect_leaf_paths(root, df, target_col, target_meaning)
    lines = ["Rule-based segmentation. No model training or train/test split is used.", ""]
    for row in rows:
        lines.append(
            f"Leaf {row['leaf_node']}: IF {row['if_conditions']} THEN {row['final_decision']} "
            f"({row['rows']} rows, {row['risk_band']})"
        )
    return "\n".join(lines)


def split_condition(condition: str) -> Tuple[str, str, str]:
    if " is not missing" in condition:
        return condition.replace(" is not missing", "").strip(), "is not missing", ""
    if " is missing" in condition:
        return condition.replace(" is missing", "").strip(), "is missing", ""
    for operator in [" <= ", " > ", " == ", " != "]:
        if operator in condition:
            left, right = condition.split(operator, 1)
            return left.strip(), operator.strip(), right.strip()
    raise ValueError(f"Could not parse rule condition: {condition}")


def python_condition(condition: str, df: pd.DataFrame) -> str:
    if " OR " in condition:
        return "(" + " or ".join(python_condition(part, df) for part in condition.split(" OR ")) + ")"
    column, operator, value = split_condition(condition)
    if operator == "is missing":
        return f"pd.isna(row.get({column!r}))"
    if operator == "is not missing":
        return f"pd.notna(row.get({column!r}))"
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column]):
        return f"pd.to_numeric(row.get({column!r}), errors='coerce') {operator} {value}"
    if operator == "==":
        return f"str(row.get({column!r}, '')) == {value!r}"
    if operator == "!=":
        return f"str(row.get({column!r}, '')) != {value!r}"
    return f"row.get({column!r}) {operator} {value}"


def sql_condition(condition: str, df: pd.DataFrame) -> str:
    if " OR " in condition:
        return "(" + " OR ".join(sql_condition(part, df) for part in condition.split(" OR ")) + ")"
    column, operator, value = split_condition(condition)
    if operator == "is missing":
        return f"[{column}] IS NULL"
    if operator == "is not missing":
        return f"[{column}] IS NOT NULL"
    sql_operator = "=" if operator == "==" else "<>" if operator == "!=" else operator
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column]):
        sql_value = value
    else:
        sql_value = sql_literal(value)
    return f"[{column}] {sql_operator} {sql_value}"


def excel_condition(condition: str, df: pd.DataFrame) -> str:
    if " OR " in condition:
        return "OR(" + ",".join(excel_condition(part, df) for part in condition.split(" OR ")) + ")"
    column, operator, value = split_condition(condition)
    if operator == "is missing":
        return f"ISBLANK([@[{column}]])"
    if operator == "is not missing":
        return f"NOT(ISBLANK([@[{column}]]))"
    excel_operator = "=" if operator == "==" else "<>" if operator == "!=" else operator
    if column in df.columns and pd.api.types.is_numeric_dtype(df[column]):
        excel_value = value
    else:
        excel_value = '"' + str(value).replace('"', '""') + '"'
    return f"[@[{column}]]{excel_operator}{excel_value}"


def export_tree_as_if_else(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    rows = collect_leaf_paths(root, df, target_col, target_meaning)
    lines = [
        "import pandas as pd",
        "",
        "def predict_from_rules(row):",
        "    \"\"\"row is a dictionary-like object with the original feature names.\"\"\"",
    ]
    for idx, row in enumerate(rows):
        prefix = "if" if idx == 0 else "elif"
        condition = row["if_conditions"]
        if condition == "All rows":
            lines.append(f"    return {row['final_decision']!r}")
            continue
        parts = [python_condition(part, df) for part in condition.split(" AND ")]
        lines.append(f"    {prefix} {' and '.join(parts)}:")
        lines.append(f"        return {row['final_decision']!r}")
    lines.append("    return None")
    return "\n".join(lines)


def sql_literal(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def export_tree_as_sql_case(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    rows = collect_leaf_paths(root, df, target_col, target_meaning)
    lines = ["CASE"]
    for row in rows:
        condition = row["if_conditions"]
        if condition == "All rows":
            condition = "1 = 1"
        else:
            condition = " AND ".join(sql_condition(part, df) for part in condition.split(" AND "))
        lines.append(f"    WHEN {condition} THEN {sql_literal(row['final_decision'])}")
    lines.append("END AS prediction")
    return "\n".join(lines)


def export_tree_as_excel_if(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    rows = collect_leaf_paths(root, df, target_col, target_meaning)
    formula = ""
    close_count = 0

    for row in rows:
        condition = row["if_conditions"]
        result = '"' + str(row["final_decision"]).replace('"', '""') + '"'
        if condition == "All rows":
            return "=" + result

        excel_condition_text = ",".join(excel_condition(part, df) for part in condition.split(" AND "))

        formula += f"IF(AND({excel_condition_text}),{result},"
        close_count += 1

    formula += '""' + (")" * close_count)
    return "=" + formula


def tree_to_base64_png(root: RuleNode, df: pd.DataFrame, target_col: str, target_meaning: str) -> str:
    class_names = sorted(df[target_col].astype(str).unique().tolist())
    good_index, bad_index = infer_good_bad_class_indices(class_names, target_col, target_meaning)
    total_rows = max(1, len(df))
    positions = {}
    leaf_order = {"value": 0}

    def assign(node: RuleNode, depth: int) -> float:
        if node.is_leaf:
            x = leaf_order["value"]
            leaf_order["value"] += 1
        else:
            left_x = assign(node.left, depth + 1)
            right_x = assign(node.right, depth + 1)
            x = (left_x + right_x) / 2
        positions[node.node_id] = (x, -depth)
        return x

    assign(root, 0)
    leaf_count = max(1, leaf_order["value"])
    depth = max(node.depth for node in iter_leaves(root))
    fig_width = max(12, min(34, leaf_count * 3.4))
    fig_height = max(7, min(24, depth * 2.8 + 4))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="#F4F9FE")

    def walk_edges(node: RuleNode) -> None:
        if node.is_leaf:
            return
        x, y = positions[node.node_id]
        for child, label, color in [
            (node.left, "Yes", "#16835B"),
            (node.right, "No", "#D71920"),
        ]:
            cx, cy = positions[child.node_id]
            ax.plot([x, cx], [y - 0.15, cy + 0.2], color="#003F7D", linewidth=1.4, alpha=0.7)
            ax.text(
                (x + cx) / 2,
                (y + cy) / 2,
                label,
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold",
                color=color,
                bbox={"boxstyle": "round,pad=0.25", "facecolor": "#FFFFFF", "edgecolor": color},
            )
            walk_edges(child)

    def node_label(node: RuleNode) -> Tuple[str, str, str]:
        stats = node_target_stats(df, node, target_col, class_names, good_index, bad_index)
        share = stats["rows"] / total_rows
        if node.is_leaf:
            text = (
                f"Leaf\n{stats['risk_band']}\nRows: {stats['rows']} ({share:.1%})\n"
                f"Class: {stats['predicted_class']}\n"
                f"Bad: {'N/A' if stats['bad_rate'] is None else format(stats['bad_rate'], '.1%')}"
            )
            edge = "#D71920" if stats["risk_band"] == "High Risk" else "#005AA9"
            fill = "#FFE5E7" if stats["risk_band"] == "High Risk" else "#DDEEFF"
            return text, fill, edge

        if node.split_operator == "is missing" or node.split_value == "__MISSING__":
            rule = f"{node.split_feature} is missing?"
        else:
            rule = f"{node.split_feature} {node.split_operator} {format_value(node.split_value)}"
        text = f"Question\n{textwrap.fill(rule, width=24)}\nRows: {stats['rows']} ({share:.1%})"
        return text, "#E7F3FF", "#003F7D"

    walk_edges(root)

    def walk_nodes(node: RuleNode) -> None:
        x, y = positions[node.node_id]
        text, fill, edge = node_label(node)
        ax.text(
            x,
            y,
            text,
            ha="center",
            va="center",
            fontsize=9.5,
            linespacing=1.3,
            bbox={"boxstyle": "round,pad=0.55", "facecolor": fill, "edgecolor": edge, "linewidth": 1.6},
            zorder=3,
        )
        if not node.is_leaf:
            walk_nodes(node.left)
            walk_nodes(node.right)

    walk_nodes(root)
    ax.set_title("Rule-Based Segmentation Tree", fontsize=18, fontweight="bold", color="#003F7D", pad=18)
    ax.text(
        0.5,
        0.985,
        "No model training is performed. Counts are actual rows from the uploaded dataset after missing target rows are removed.",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=10.5,
        color="#65758B",
    )
    ax.set_xlim(-0.75, leaf_count - 0.25)
    ax.set_ylim(-depth - 0.8, 0.85)
    ax.axis("off")

    buffer = io.BytesIO()
    fig.tight_layout()
    fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")


def validate_tree_limits(max_depth: int, max_leaf_nodes: int) -> None:
    if max_depth < 1:
        raise ValueError("Max tree depth must be at least 1.")
    if max_leaf_nodes < 2:
        raise ValueError("Max leaf nodes must be at least 2.")
    if max_leaf_nodes > 2 ** max_depth:
        raise ValueError(
            f"Max leaf nodes cannot be {max_leaf_nodes} when max tree depth is {max_depth}. "
            f"With depth {max_depth}, the tree can have at most {2 ** max_depth} leaf nodes."
        )


def analyze_and_render(form):
    dataset_id = form.get("dataset_id", "").strip()
    if not dataset_id:
        raise ValueError("Please upload a dataset first.")

    original_df = load_clean_dataset(dataset_id)
    original_row_count = len(original_df)
    target_col = form.get("target_col", "").strip()
    target_meaning = form.get("target_meaning", "auto").strip() or "auto"
    feature_cols = [col for col in form.getlist("feature_cols") if col]
    lower_cap_value = float(form.get("lower_cap_value", 2.5))
    upper_cap_value = float(form.get("upper_cap_value", 97.5))
    max_depth = int(form.get("max_depth", 4))
    max_leaf_nodes = int(form.get("max_leaf_nodes", 12))
    categorical_unique_threshold = int(form.get("categorical_unique_threshold", 10))

    validate_tree_limits(max_depth, max_leaf_nodes)

    if target_col not in original_df.columns:
        raise ValueError(f"Target '{target_col}' not found.")

    df = original_df.dropna(subset=[target_col]).copy()
    rows_removed_missing_target = original_row_count - len(df)
    if df.empty:
        raise ValueError("No rows remain after removing blank target values.")

    if feature_cols:
        valid_feature_cols = [c for c in feature_cols if c in df.columns and c != target_col]
    else:
        valid_feature_cols = [c for c in df.columns if c != target_col]

    if not valid_feature_cols:
        raise ValueError("Please select at least one valid feature column.")

    df = df[[target_col] + valid_feature_cols]
    continuous_cols, categorical_cols, numeric_cols = identify_variable_types(df, target_col, categorical_unique_threshold)
    df, outlier_report = cap_outliers(df, continuous_cols, lower_cap_value, upper_cap_value)
    root = build_rule_tree(df, target_col, valid_feature_cols, max_depth, max_leaf_nodes)
    leaf_row_total = sum(len(leaf.row_indices) for leaf in iter_leaves(root))
    unique_leaf_rows = len({idx for leaf in iter_leaves(root) for idx in leaf.row_indices})
    if leaf_row_total != len(df) or unique_leaf_rows != len(df):
        raise ValueError(
            "Segmentation row audit failed: leaf rows do not exactly match analyzed rows. "
            "Please check duplicate dataframe indexes or malformed input data."
        )
    rules = export_rules_text(root, df, target_col, target_meaning)
    if_else_rules = export_tree_as_if_else(root, df, target_col, target_meaning)
    sql_case_rules = export_tree_as_sql_case(root, df, target_col, target_meaning)
    excel_if_rules = export_tree_as_excel_if(root, df, target_col, target_meaning)

    row_audit = pd.DataFrame(
        [
            {"stage": "Original uploaded dataset", "rows": original_row_count},
            {"stage": "Rows removed because target is blank", "rows": rows_removed_missing_target},
            {"stage": "Rows analyzed by rule engine", "rows": len(df)},
            {"stage": "Rows assigned to final leaves", "rows": leaf_row_total},
            {"stage": "Unique rows assigned to final leaves", "rows": unique_leaf_rows},
            {"stage": "Rows used for model training", "rows": 0},
            {"stage": "Rows used for train/test split", "rows": 0},
        ]
    )

    return {
        "mode_note": "Rule-based segmentation only. No model training, no train/test split, and no weighted row counts.",
        "accuracy": "N/A - no model training",
        "precision": "N/A - no model training",
        "model_gini": "N/A - no trained model",
        "tree_depth": str(max_depth),
        "leaf_nodes": str(len(iter_leaves(root))),
        "encoded_features": len(valid_feature_cols),
        "row_audit_table": row_audit.to_html(index=False, classes="table"),
        "tree_image": tree_to_base64_png(root, df, target_col, target_meaning),
        "rules": rules,
        "rules_b64": text_to_base64(rules),
        "if_else_rules": if_else_rules,
        "if_else_rules_b64": text_to_base64(if_else_rules),
        "sql_case_rules": sql_case_rules,
        "sql_case_rules_b64": text_to_base64(sql_case_rules),
        "excel_if_rules": excel_if_rules,
        "excel_if_rules_b64": text_to_base64(excel_if_rules),
        "outlier_table": outlier_report.to_html(index=False, classes="table"),
        "continuous_table": make_table(continuous_cols, "Variable"),
        "categorical_table": make_table(categorical_cols, "Variable"),
        "numeric_table": make_table(numeric_cols, "Variable"),
        "leaf_summary_table": make_leaf_summary_table(root, df, target_col, target_meaning),
        "leaf_paths_table": pd.DataFrame(collect_leaf_paths(root, df, target_col, target_meaning)).to_html(
            index=False, classes="table"
        ),
    }


PAGE_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Rule-Based Decision Tree Analyzer</title>
  <style>
    :root {
      --tata-blue: #005AA9;
      --tata-deep-blue: #003F7D;
      --tata-red: #D71920;
      --bg: #F4F9FE;
      --panel: #FFFFFF;
      --ink: #172033;
      --muted: #65758B;
      --line: #D8E4F0;
    }
    * { box-sizing: border-box; }
    body {
      margin: 0;
      font-family: Arial, Helvetica, sans-serif;
      background: var(--bg);
      color: var(--ink);
    }
    header {
      background: var(--tata-deep-blue);
      color: white;
      padding: 22px 28px;
    }
    header h1 { margin: 0 0 6px; font-size: 26px; }
    header p { margin: 0; color: #DDEEFF; }
    main { max-width: 1240px; margin: 0 auto; padding: 24px; }
    section, form.panel {
      background: var(--panel);
      border: 1px solid var(--line);
      border-radius: 8px;
      padding: 18px;
      margin-bottom: 18px;
    }
    h2 { margin: 0 0 14px; font-size: 20px; color: var(--tata-deep-blue); }
    h3 { margin: 18px 0 10px; font-size: 16px; color: var(--tata-deep-blue); }
    .grid { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 14px; }
    label { display: block; font-weight: 700; margin-bottom: 6px; }
    input, select {
      width: 100%;
      padding: 9px 10px;
      border: 1px solid var(--line);
      border-radius: 6px;
      font-size: 14px;
      background: white;
    }
    select[multiple] { min-height: 180px; }
    button, .download {
      display: inline-block;
      border: 0;
      border-radius: 6px;
      background: var(--tata-blue);
      color: white;
      padding: 10px 14px;
      font-weight: 700;
      cursor: pointer;
      text-decoration: none;
      margin: 4px 6px 4px 0;
    }
    .secondary { background: #65758B; }
    .danger { background: var(--tata-red); }
    .notice {
      border-left: 4px solid var(--tata-blue);
      background: #E7F3FF;
      padding: 12px 14px;
      margin-bottom: 14px;
    }
    .error {
      border-left: 4px solid var(--tata-red);
      background: #FFE5E7;
      padding: 12px 14px;
      margin-bottom: 14px;
    }
    .table-scroll { overflow-x: auto; }
    table.table, .dataframe {
      width: 100%;
      border-collapse: collapse;
      font-size: 13px;
      background: white;
    }
    table.table th, table.table td, .dataframe th, .dataframe td {
      border: 1px solid var(--line);
      padding: 8px 10px;
      text-align: left;
    }
    table.table th, .dataframe th {
      background: #E7F3FF;
      color: var(--tata-deep-blue);
    }
    pre {
      white-space: pre-wrap;
      background: #0F172A;
      color: #E5E7EB;
      padding: 14px;
      border-radius: 8px;
      overflow-x: auto;
      font-size: 13px;
    }
    img.tree { width: 100%; height: auto; border: 1px solid var(--line); border-radius: 8px; background: white; }
    .metrics { display: grid; grid-template-columns: repeat(5, minmax(0, 1fr)); gap: 10px; }
    .metric { border: 1px solid var(--line); border-radius: 8px; padding: 10px; background: #FBFDFF; }
    .metric strong { display: block; color: var(--muted); font-size: 12px; margin-bottom: 4px; }
    @media (max-width: 800px) {
      main { padding: 14px; }
      .grid, .metrics { grid-template-columns: 1fr; }
    }
  </style>
</head>
<body>
  <header>
    <h1>Rule-Based Decision Tree Analyzer</h1>
    <p>Dataset-independent workflow: uploaded data is segmented, not used to train a model.</p>
  </header>
  <main>
    {% if error %}<div class="error">{{ error }}</div>{% endif %}

    <form class="panel" method="post" enctype="multipart/form-data">
      <h2>Upload Dataset</h2>
      <input type="hidden" name="action" value="load_columns">
      <input type="file" name="dataset" accept=".csv,.xlsx,.xls">
      <div style="margin-top:12px;">
        <button type="submit">Load Columns</button>
      </div>
    </form>

    {% if columns %}
    <form class="panel" method="post">
      <input type="hidden" name="action" value="analyze">
      <input type="hidden" name="dataset_id" value="{{ dataset_id }}">
      <h2>Configure Segmentation</h2>
      <div class="notice">No model is trained. The target is used only to summarize each segment's distribution and risk rate.</div>
      <div class="grid">
        <div>
          <label>Target Column</label>
          <select name="target_col" required>
            {% for col in columns %}<option value="{{ col }}">{{ col }}</option>{% endfor %}
          </select>
        </div>
        <div>
          <label>Target Meaning</label>
          <select name="target_meaning">
            <option value="auto">Auto Detect</option>
            <option value="high_bad">1 / Yes / Higher class is Bad</option>
            <option value="high_good">1 / Yes / Higher class is Good</option>
          </select>
        </div>
        <div>
          <label>Feature Columns</label>
          <select name="feature_cols" multiple>
            {% for col in columns %}<option value="{{ col }}">{{ col }}</option>{% endfor %}
          </select>
        </div>
        <div class="grid">
          <div>
            <label>Max Depth</label>
            <input type="number" name="max_depth" value="4" min="1" max="8">
          </div>
          <div>
            <label>Max Leaf Nodes</label>
            <input type="number" name="max_leaf_nodes" value="12" min="2" max="64">
          </div>
          <div>
            <label>Lower Cap Percentile</label>
            <input type="number" step="0.1" name="lower_cap_value" value="2.5" min="0" max="100">
          </div>
          <div>
            <label>Upper Cap Percentile</label>
            <input type="number" step="0.1" name="upper_cap_value" value="97.5" min="0" max="100">
          </div>
          <div>
            <label>Categorical Unique Threshold</label>
            <input type="number" name="categorical_unique_threshold" value="10" min="2" max="100">
          </div>
        </div>
      </div>
      <div style="margin-top:14px;">
        <button type="submit">Analyze Without Training</button>
        <button class="secondary" name="action" value="reset" type="submit">Reset</button>
      </div>
    </form>

    <section>
      <h2>Dataset Preview</h2>
      <div class="table-scroll">{{ dataset_preview|safe }}</div>
    </section>
    {% endif %}

    {% if result %}
    <section>
      <h2>Results</h2>
      <div class="notice">{{ result.mode_note }}</div>
      <div class="metrics">
        <div class="metric"><strong>Accuracy</strong>{{ result.accuracy }}</div>
        <div class="metric"><strong>Precision</strong>{{ result.precision }}</div>
        <div class="metric"><strong>Gini</strong>{{ result.model_gini }}</div>
        <div class="metric"><strong>Depth Setting</strong>{{ result.tree_depth }}</div>
        <div class="metric"><strong>Leaf Nodes</strong>{{ result.leaf_nodes }}</div>
      </div>
    </section>

    <section>
      <h2>Row Audit</h2>
      <div class="table-scroll">{{ result.row_audit_table|safe }}</div>
    </section>

    <section>
      <h2>Segmentation Tree</h2>
      <img class="tree" src="data:image/png;base64,{{ result.tree_image }}" alt="Rule tree">
    </section>

    <section>
      <h2>Leaf Summary</h2>
      <div class="table-scroll">{{ result.leaf_summary_table|safe }}</div>
    </section>

    <section>
      <h2>Leaf Paths</h2>
      <div class="table-scroll">{{ result.leaf_paths_table|safe }}</div>
    </section>

    <section>
      <h2>Variable Types</h2>
      <div class="grid">
        <div><h3>Continuous</h3><div class="table-scroll">{{ result.continuous_table|safe }}</div></div>
        <div><h3>Categorical</h3><div class="table-scroll">{{ result.categorical_table|safe }}</div></div>
        <div><h3>Numeric Category</h3><div class="table-scroll">{{ result.numeric_table|safe }}</div></div>
        <div><h3>Outlier Capping</h3><div class="table-scroll">{{ result.outlier_table|safe }}</div></div>
      </div>
    </section>

    <section>
      <h2>Exports</h2>
      <a class="download" download="rules.txt" href="data:text/plain;base64,{{ result.rules_b64 }}">Download Rules</a>
      <a class="download" download="rules.py" href="data:text/plain;base64,{{ result.if_else_rules_b64 }}">Download Python</a>
      <a class="download" download="rules.sql" href="data:text/plain;base64,{{ result.sql_case_rules_b64 }}">Download SQL</a>
      <a class="download" download="rules_excel.txt" href="data:text/plain;base64,{{ result.excel_if_rules_b64 }}">Download Excel IF</a>
      <h3>Plain Rules</h3><pre>{{ result.rules }}</pre>
      <h3>Python If/Else</h3><pre>{{ result.if_else_rules }}</pre>
      <h3>SQL Case</h3><pre>{{ result.sql_case_rules }}</pre>
      <h3>Excel IF</h3><pre>{{ result.excel_if_rules }}</pre>
    </section>
    {% endif %}
  </main>
</body>
</html>
"""


@app.route("/", methods=["GET", "POST"])
def index():
    error = None
    result = None
    columns = None
    dataset_id = None
    dataset_preview = None

    if request.method == "POST":
        action = request.form.get("action", "load_columns")
        if action == "reset":
            return render_template_string(PAGE_TEMPLATE, error=None, result=None, columns=None, dataset_id=None)

        if action == "load_columns":
            uploaded_file = request.files.get("dataset")
            if not uploaded_file or uploaded_file.filename == "":
                error = "Please upload a CSV or Excel dataset."
            else:
                try:
                    dataset_id = save_uploaded_dataset(uploaded_file)
                    columns = get_dataset_columns(dataset_id)
                    dataset_preview = get_dataset_preview(dataset_id)
                except Exception as exc:
                    error = str(exc)

        elif action == "analyze":
            dataset_id = request.form.get("dataset_id", "").strip()
            try:
                columns = get_dataset_columns(dataset_id)
                dataset_preview = get_dataset_preview(dataset_id)
                result = analyze_and_render(request.form)
            except Exception as exc:
                if dataset_id:
                    try:
                        columns = columns or get_dataset_columns(dataset_id)
                        dataset_preview = dataset_preview or get_dataset_preview(dataset_id)
                    except Exception:
                        pass
                error = str(exc)
        else:
            error = "Unknown action. Please upload the dataset again."

    return render_template_string(
        PAGE_TEMPLATE,
        error=error,
        result=result,
        columns=columns,
        dataset_id=dataset_id,
        dataset_preview=dataset_preview,
    )


def open_chrome_browser(url: str) -> None:
    chrome_paths = [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        os.path.expandvars(r"%LOCALAPPDATA%\Google\Chrome\Application\chrome.exe"),
    ]

    try:
        webbrowser.get("chrome").open_new(url)
        return
    except webbrowser.Error:
        pass

    for chrome_path in chrome_paths:
        if os.path.exists(chrome_path):
            webbrowser.register("chrome", None, webbrowser.BackgroundBrowser(chrome_path))
            webbrowser.get("chrome").open_new(url)
            return

    webbrowser.open_new(url)


def open_browser_after_start(port: int) -> None:
    url = f"http://localhost:{port}"
    threading.Timer(1.5, open_chrome_browser, args=(url,)).start()


if __name__ == "__main__":
    port = int(os.environ.get("PORT", "5000"))
    open_browser_after_start(port)
    app.run(host="127.0.0.1", port=port, debug=False, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [15/Jul/2026 10:02:02] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [15/Jul/2026 10:02:03] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [15/Jul/2026 10:02:03] "GET /favicon.ico HTTP/1.1" 404 -


127.0.0.1 - - [15/Jul/2026 10:02:20] "GET / HTTP/1.1" 200 -
